# RCAEval Exploration

Goal: look at real failure shapes before designing Step 1 (anomaly/symptom detection).

Specifically: is a `mem` fault gradual (leak-like) or a step-function jump? Is `delay`/`loss` genuinely sudden? We look at full metric series, not windows, so we don't accidentally throw away the shape we're trying to study.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Settled functions (data access, shape/evidence rules, call graph, log templates, step 0, scoring) live in rca_lib.py.
from rca_lib import *

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

idx = load_index()  # RCAEval-data/cases.parquet (read-only)
idx.head()

## 1. Pick one case per fault type to compare shapes

Start with `mem` (suspected gradual) vs `delay` or `loss` (suspected sudden), same system, same root-cause service if possible, so we're comparing fault shape and not service-specific noise.

In [ ]:
candidates = idx[(idx.dataset == "RE1-OB") & (idx.root_cause_service == "adservice")]
candidates[["case", "fault", "fault_description", "duration_minutes", "n_timesteps"]]

In [ ]:
# load_metrics, load_inject_time, latency_col: rca_lib.py
# Fill these in with real case names from the cell above once you've picked them
mem_case = "re1ob_adservice_mem_1"
delay_case = "re1ob_adservice_delay_1"

mem_metrics = load_metrics(mem_case)
mem_inject = load_inject_time(mem_case)

delay_metrics = load_metrics(delay_case)
delay_inject = load_inject_time(delay_case)

print(mem_case, "columns:", [c for c in mem_metrics.columns if "adservice" in c])

## 2. Plot the FULL series (no windowing) for the root-cause service's relevant metric

This is the whole point: don't pre-select a window and risk cutting off the lead-up. Look at the entire case duration.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

axes[0].plot(mem_metrics["time"], mem_metrics["adservice_mem"])
axes[0].axvline(mem_inject, color="red", linestyle="--", label="inject_time")
axes[0].set_title(f"{mem_case} — adservice_mem (full series)")
axes[0].legend()

axes[1].plot(delay_metrics["time"], delay_metrics[latency_col(delay_metrics)])
axes[1].axvline(delay_inject, color="red", linestyle="--", label="inject_time")
axes[1].set_title(f"{delay_case} — {latency_col(delay_metrics)} (full series)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Quantify the shape: step-change vs gradual drift

`shape_signature_v1` is the original (300 s before / 60 s after, raw units) and is kept **only** so the comparison table below can show what changed. Its problems:
- `pre_slope` looks *before* inject_time, so it can't see a fault that ramps up *after* injection. At best it checks whether inject_time is right.
- `step_score` divides by the pre-window std. On near-flat baselines (latency) that gives 3,000-9,000, and neither number compares across metrics or units.
- 300 s / 60 s are arbitrary. A slow leak barely moves in 60 s.

`shape_signature_v2`:
- **Windows from each case's data.** Baseline = the whole normal period, minus the first 30 s (some series start with a collection spike). Fault = the whole faulty period. These equal `normal_timesteps` / `faulty_timesteps` in `cases.parquet` (1 row/s).
- **Unit-free outputs.** `total_z` (shift in baseline-noise units) answers "is the shift real?". `log2_fold` answers "how big is it?" and compares across metrics. The shape measures are *shares of the final shift*, so they're dimensionless.
- **Post-injection shape.** `late_trend_share` (does it keep moving after the initial response?) and `time_to_plateau_s` (when does it first reach 90% of its final level?).
- **`pre_drift_share` replaces `pre_slope`**, and is only a sanity check on inject_time.
- **One measured constant.** `RESPONSE_LAG_S = 60`. Even an instantaneous fault (tc-netem delay) takes 36-57 s to fully show in `latency-90`, and cpu takes ~50 s. So "reaches its level within ~60 s" is as step-like as these metrics can look.

In [ ]:
import numpy as np

def shape_signature_v1(metrics_df, col, inject_time, pre_window=300, post_window=60):
    """ORIGINAL version - kept only for the old-vs-new comparison. Use shape_signature_v2."""
    pre = metrics_df[(metrics_df.time >= inject_time - pre_window) & (metrics_df.time < inject_time)]
    post = metrics_df[(metrics_df.time >= inject_time) & (metrics_df.time < inject_time + post_window)]

    if len(pre) < 2 or len(post) < 2:
        return None

    pre_std = pre[col].std() or 1e-9
    step_score = abs(post[col].mean() - pre[col].mean()) / pre_std

    x = pre["time"].values.astype(float)
    y = pre[col].values.astype(float)
    slope = np.polyfit(x - x[0], y, 1)[0] if len(x) > 1 else 0

    return {"step_score": step_score, "pre_slope": slope}

# shape_signature_v2, classify_shape, THRESHOLDS, presence/sparse-rate signatures and metric_evidence: rca_lib.py

print("mem case:  ", shape_signature_v2(mem_metrics, "adservice_mem", mem_inject))
print("delay case:", shape_signature_v2(delay_metrics, latency_col(delay_metrics), delay_inject))

**Reading the v2 columns:**
- `log2_fold`: size of the final shift. 1 = doubled, 5.9 ≈ 60×, 0.06 ≈ +4%. Compare this across metrics, not `total_z`. `total_z` is large whenever the baseline is quiet.
- `early_share`: how much of the final shift is already there within `RESPONSE_LAG_S` of injection. It is <1 even for true steps because of the metric lag, so read it together with `time_to_plateau_s`.
- `late_trend_share`: ≈0 means it jumped and held. ≈1 means it kept climbing by about the whole shift again over the rest of the fault. **This is the leak detector.**
- `time_to_plateau_s` ≤ ~120 s (2 × response lag) is step-like. Much longer means gradual, or a late level change.
- `pre_drift_share`: should be ≈0. A large value means the baseline was already moving before inject_time. It is unreliable when the shift itself is tiny, because the denominator is small.

- `time_to_plateau_s` is measured against the **median of the fault period** (after `RESPONSE_LAG_S`), not the end level, so a late surge doesn't push the plateau out.

`classify_shape` is a provisional first cut of these rules. A shift passes the size check by the fold route (`log2_fold`) or, for slow leaks, by the trend route: high `total_z` **and** still moving steadily late in the fault (`late_trend_share`, `late_trend_rho`). High z alone never passes. The thresholds are fixed in `THRESHOLDS` and deliberately **not tuned** on these 25 cases. If any threshold on the decision path is within 20% (`BORDERLINE_MARGIN`), the case is labelled `borderline`: `lean` gives the label the rule would otherwise assign, and `near_threshold` says which threshold was close.

In [ ]:
results = []
for _, row in candidates.iterrows():
    case, fault = row["case"], row["fault"]
    m = load_metrics(case)
    t = load_inject_time(case)
    metric_col = f"adservice_{fault}" if fault in ("mem", "cpu") else latency_col(m)
    if metric_col not in m.columns:
        print(f"SKIP {case}: no column {metric_col}")
        continue
    old = shape_signature_v1(m, metric_col, t) or {}
    new = shape_signature_v2(m, metric_col, t)
    results.append({"case": case, "fault": fault, "metric": metric_col,
                    "OLD step_score": old.get("step_score"), "OLD pre_slope": old.get("pre_slope"),
                    **new})
    results[-1]["shape"], results[-1]["lean"], results[-1]["near_threshold"] = classify_shape(new)

compare = pd.DataFrame(results).drop(columns=["plateau_frac", "normal_s", "fault_s"])
compare.round(3)

In [ ]:
# Visual check of the table: every candidate's full series, with inject_time (red) and time_to_plateau (green)
fig, axes = plt.subplots(5, 5, figsize=(22, 16))
for ax, r in zip(axes.flat, results):
    m = load_metrics(r["case"])
    t = load_inject_time(r["case"])
    ax.plot(m.time - t, m[r["metric"]], lw=0.5)
    ax.axvline(0, color="red", lw=1)
    if not np.isnan(r["time_to_plateau_s"]):
        ax.axvline(r["time_to_plateau_s"], color="green", lw=1, ls="--")
    ax.set_title(f'{r["case"]} | {r["metric"]} | {r["shape"]}', fontsize=8)
plt.tight_layout()
plt.show()

### 3a. Does the gradual rule actually fire? Synthetic ramps

None of the 25 real cases turned out gradual, so the rule is untested on its positive class. Build synthetic leaks from real data: take `re1ob_adservice_mem_1`'s normal period (2,100 s of real `adservice_mem` noise), repeat it as the fault period, and add a linear ramp starting at inject_time. The ramp speeds are chosen by how far the ramp has climbed by the end of the fault period: +10% (very slow), +20% (slow), +100%, +1000% (reaching about the size of the real mem faults).

In [ ]:
base_case = "re1ob_adservice_mem_1"
bm = load_metrics(base_case)
b_inject = load_inject_time(base_case)
normal = bm[bm.time < b_inject][["time", "adservice_mem"]].reset_index(drop=True)
baseline_level = normal["adservice_mem"].median()

synthetic = []
for end_rise in [0.1, 0.2, 1.0, 10.0]:
    fault = normal.copy()
    fault["time"] = normal["time"] + len(normal)  # continues right after the normal period
    t_rel = fault["time"] - b_inject
    ramp_per_s = end_rise * baseline_level / t_rel.iloc[-1]
    fault["adservice_mem"] = fault["adservice_mem"] + ramp_per_s * t_rel
    synth = pd.concat([normal, fault], ignore_index=True)
    sig = shape_signature_v2(synth, "adservice_mem", b_inject)
    shape, lean, near = classify_shape(sig)
    synthetic.append({"ramp_to_end": f"+{end_rise:.0%}", "ramp_bytes_per_s": ramp_per_s, **sig,
                      "shape": shape, "lean": lean, "near_threshold": near, "_series": synth})

pd.DataFrame(synthetic).drop(columns=["_series", "normal_s", "fault_s"]).round(3)

In [ ]:
fig, axes = plt.subplots(1, len(synthetic), figsize=(18, 3.5))
for ax, r in zip(axes, synthetic):
    s = r["_series"]
    ax.plot(s.time - b_inject, s["adservice_mem"], lw=0.5)
    ax.axvline(0, color="red", lw=1)
    if not np.isnan(r["time_to_plateau_s"]):
        ax.axvline(r["time_to_plateau_s"], color="green", lw=1, ls="--")
    ax.set_title(f'synthetic ramp {r["ramp_to_end"]} | {r["shape"]} ({r["lean"]})', fontsize=9)
plt.tight_layout()
plt.show()

### 3b. Plateau reference: end level vs fault-period median (disk cases)

`plateau_ref="late"` is the Part A behaviour (90% of the final-25% level). `"fault_median"` is the new default.

In [ ]:
rows = []
for case in ["re1ob_adservice_disk_1", "re1ob_adservice_disk_3", "re1ob_adservice_disk_5"]:
    m = load_metrics(case)
    t = load_inject_time(case)
    col = latency_col(m)
    for ref in ["late", "fault_median"]:
        sig = shape_signature_v2(m, col, t, plateau_ref=ref)
        shape, lean, near = classify_shape(sig)
        rows.append({"case": case, "plateau_ref": ref, "time_to_plateau_s": sig["time_to_plateau_s"],
                     "late_trend_share": sig["late_trend_share"], "shape": shape, "lean": lean, "near_threshold": near})
pd.DataFrame(rows).round(3)

### 3c. Detection limit for a 360 s fault period

Most RE1-OB latency cases (and many others) have 360 s normal + 360 s faulty periods. Find the smallest linear ramp the rule labels `gradual`, on two real baselines: `adservice_mem` (quiet: from the 5 mem cases' normal periods) and `adservice_latency-90` (noisy: from the 5 disk cases' normal periods). The fault-period noise is the real normal-period noise, repeated, plus the ramp. "Rise" = how far the ramp has climbed by the end of the 360 s.

In [ ]:
FAULT_S = 360
RISES = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5, 0.75, 1.0, 2.0]

def synth_ramp(normal_vals, rise, fault_s=FAULT_S):
    """normal_vals: real baseline values at 1 s. Returns (df, inject) with a normal period of
    len(normal_vals) and a fault period of fault_s = baseline noise (repeated) + linear ramp."""
    n = len(normal_vals)
    noise = np.resize(normal_vals, fault_s + 1)
    level = np.median(normal_vals)
    ramp = rise * level * np.arange(fault_s + 1) / fault_s
    vals = np.concatenate([normal_vals, noise + ramp])
    return pd.DataFrame({"time": np.arange(len(vals)), "v": vals}), n

sweep = []
for metric, cases in [("adservice_mem", [f"re1ob_adservice_mem_{i}" for i in range(1, 6)]),
                      ("adservice_latency-90", [f"re1ob_adservice_disk_{i}" for i in range(1, 6)])]:
    for case in cases:
        m = load_metrics(case)
        t = load_inject_time(case)
        # last 360 s before inject + STARTUP_SKIP_S, so the baseline window is 360 s after the skip
        normal_vals = m[m.time < t][metric].values[-(FAULT_S + STARTUP_SKIP_S):]
        for rise in RISES:
            df, inj = synth_ramp(normal_vals, rise)
            sig = shape_signature_v2(df, "v", inj)
            shape, lean, near = classify_shape(sig)
            sweep.append({"metric": metric, "baseline_from": case, "rise": rise, "total_z": sig["total_z"],
                          "log2_fold": sig["log2_fold"], "late_trend_share": sig["late_trend_share"],
                          "late_trend_rho": sig["late_trend_rho"], "shape": shape, "lean": lean, "near": near})
sweep = pd.DataFrame(sweep)

# label per baseline x rise ("B:" = borderline, followed by the lean)
lab = sweep.assign(label=np.where(sweep["shape"] == "borderline", "B:" + sweep["lean"], sweep["shape"]))
lab.pivot_table(index=["metric", "baseline_from"], columns="rise", values="label", aggfunc="first")

In [ ]:
def smallest(g, ok):
    hits = g[ok(g)].rise
    return hits.min() if len(hits) else np.nan

limit = sweep.groupby(["metric", "baseline_from"]).apply(lambda g: pd.Series({
    "smallest_rise_gradual": smallest(g, lambda x: x["shape"] == "gradual"),
    "smallest_rise_gradual_or_borderline": smallest(g, lambda x: x["lean"] == "gradual"),
}), include_groups=False)
limit

## 4. Shape across all RE1 and RE2 cases

Same `shape_signature_v2` / `classify_shape`, with thresholds unchanged. Choice of metric per fault type:

| fault | metric (first that exists) |
|---|---|
| cpu | `{svc}_cpu` |
| mem | `{svc}_mem` |
| delay | `{svc}_latency-90` → `{svc}_latency` → `{svc}_latency-50` |
| disk | `{svc}_diskio` (RE2 only) → latency fallback (RE1 has no diskio) |
| loss | latency, as above (plus the error/caller comparison in 4b) |
| socket | `{svc}_socket` (RE2 only). Explicit: no fallback, because a socket fault is about socket counts, not latency |

`load`/`workload` is resolved the same way (`workload` → `load`) but isn't the primary metric for any fault type.

Nothing is skipped silently. Every excluded or unusable case is listed with a reason.

In [ ]:
part_b, skipped = [], []
for _, r in idx[idx.suite.isin(["RE1", "RE2"])].iterrows():
    case, svc, fault = r["case"], r["root_cause_service"], r["fault"]
    if case in EXCLUDE:
        skipped.append({"case": case, "reason": "excluded: " + EXCLUDE[case]})
        continue
    m = load_metrics(case)
    t = load_inject_time(case)
    col, note = pick_fault_metric(m, svc, fault)
    if col is None:
        skipped.append({"case": case, "reason": note})
        continue
    ev = metric_evidence(m, col, t, sparse=False)
    if ev["shape"] == "no_data":
        # no usable baseline or fault period for the shape - but the metric appearing/vanishing IS evidence
        if ev["presence_change"] == "none":
            skipped.append({"case": case, "reason": f"{col}: no usable data on either side of inject_time and no presence change"})
            continue
        ev["shape"] = ev["lean"] = ev["presence_change"]
    part_b.append({"case": case, "dataset": r["dataset"], "service": svc, "fault": fault, "metric_note": note, **ev})
part_b = pd.DataFrame(part_b)
skipped = pd.DataFrame(skipped, columns=["case", "reason"])

print(f"analysed {len(part_b)} cases, skipped {len(skipped)}")
print("presence-only labels (shape not computable, metric appears/goes quiet):",
      part_b[part_b["shape"].isin(["appears", "goes_quiet"])].case.tolist())
print(skipped.to_string(index=False))
print("\nmetric fallbacks used:")
print(part_b[part_b.metric_note != ""].groupby(["dataset", "fault", "metric_note"]).size().to_string())

In [ ]:
# Per fault type: how many cases look step-like vs gradual. Borderline cases are counted separately, split by lean.
part_b["label"] = np.where(part_b["shape"] == "borderline", "borderline:" + part_b["lean"], part_b["shape"])
order = ["step", "gradual", "receding", "no_clear_shift", "appears", "goes_quiet",
         "borderline:step", "borderline:gradual", "borderline:receding", "borderline:no_clear_shift"]
summary = pd.crosstab(part_b.fault, part_b.label).reindex(columns=order, fill_value=0)
summary["total"] = summary.sum(axis=1)
# presence change on the primary metric, independent of its shape label (e.g. latency samples vanishing under loss)
summary["+presence_change"] = part_b[part_b.presence_change != "none"].groupby("fault").size().reindex(summary.index, fill_value=0)
summary

In [ ]:
# Same table by dataset, to see whether a fault type behaves differently per system / suite
by_ds = pd.crosstab([part_b.fault, part_b.dataset], part_b.label).reindex(columns=order, fill_value=0)
by_ds

In [ ]:
# Every case labelled gradual (clear or leaning) - these are the ones to eyeball
grad = part_b[part_b.lean.isin(["gradual", "receding"])].sort_values(["fault", "dataset"])
grad[["case", "metric", "total_z", "log2_fold", "late_trend_share", "late_trend_rho", "time_to_plateau_s", "shape", "lean", "near_threshold"]].round(3)

In [ ]:
# Presence change on the primary metric, by fault and direction (counted on top of the shape label)
pd.crosstab(part_b.fault, part_b.presence_change)

In [ ]:
fig, axes = plt.subplots(int(np.ceil(len(grad) / 5)), 5, figsize=(22, 3.2 * np.ceil(len(grad) / 5)), squeeze=False)
for ax, (_, r) in zip(axes.flat, grad.iterrows()):
    m = load_metrics(r["case"])
    t = load_inject_time(r["case"])
    ax.plot(m.time - t, m[r["metric"]], lw=0.5)
    ax.axvline(0, color="red", lw=1)
    if not np.isnan(r["time_to_plateau_s"]):
        ax.axvline(r["time_to_plateau_s"], color="green", lw=1, ls="--")
    ax.set_title(f'{r["case"]} | {r["metric"]} | {r["shape"]} ({r["lean"]})', fontsize=7)
for ax in axes.flat[len(grad):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

### 4b. Loss faults: which metric shows the shift?

For each loss case, run the same signature on:
- **own latency**: `{svc}_latency-90` (the 4a choice)
- **own error**: `{svc}_error`, only where the column exists. It is missing in many cases, and missing is **not** treated as zero.
- **caller latency**: latency-90 of each service that calls the root-cause service. The call graph comes from RE2/RE3 traces for OB and TT (`src=trace`). Services the traces don't cover (OB adservice/cartservice/shippingservice, all of Sock Shop, which has no traces) come from the systems' published architectures (`src=static`). ts-auth-service has no caller in traces and none added.

"Shows the shift" = passes the size check (lean is not `no_clear_shift`). For callers, the best caller per case is taken.

In [ ]:
# trace_edges, STATIC_EDGES, callers_of: rca_lib.py (trace edges are computed lazily and cached)
for system, svcs in idx[idx.suite.isin(["RE1", "RE2"]) & (idx.fault == "loss")].groupby("system").root_cause_service.unique().items():
    for s in svcs:
        print(f"{system} {s:24s} callers: {callers_of(system, s)}")

In [ ]:
loss_rows = []
for _, r in idx[idx.suite.isin(["RE1", "RE2"]) & (idx.fault == "loss")].iterrows():
    case, svc = r["case"], r["root_cause_service"]
    if case in EXCLUDE:
        continue  # already listed in the skipped table above
    m = load_metrics(case)
    t = load_inject_time(case)
    kinds = [("own_latency", resolve_col(m, svc, "latency"), ""),
             ("own_error", resolve_col(m, svc, "error"), "")]
    kinds += [("caller_latency", resolve_col(m, c, "latency"), f"{c} ({src})") for c, src in callers_of(r["system"], svc)]
    for kind, col, via in kinds:
        row = {"case": case, "dataset": r["dataset"], "service": svc, "kind": kind, "via": via}
        if col is None:
            row.update({"metric": None, "lean": "column_missing", "shifted": False, "evidence": ""})
        else:
            row.update(metric_evidence(m, col, t))  # error -> sparse-rate view; all -> presence view
        loss_rows.append(row)
loss_df = pd.DataFrame(loss_rows)
loss_df["detected"] = loss_df.shifted.astype(bool)

# per case, one row per kind; for callers keep the most clearly shifted one (largest |log2_fold| among detected, else any)
best = (loss_df.assign(_k=loss_df.detected.astype(int) * 1e6 + loss_df.log2_fold.abs().fillna(-1))
        .sort_values("_k", ascending=False).groupby(["case", "kind"]).head(1).drop(columns="_k"))

no_caller = sorted(set(loss_df.case) - set(loss_df[loss_df.kind == "caller_latency"].case))
print(f"loss cases with no known caller (no caller_latency row): {len(no_caller)} -> {sorted({c.rsplit('_', 2)[0] for c in no_caller})}")

def rate(g):
    have = g[g.lean != "column_missing"]
    ev = have[have.detected].evidence.str.split(",").explode()
    return pd.Series({"cases": len(g), "metric_available": len(have), "detected": int(have.detected.sum()),
                      "weak_only": int(have.weak.fillna(False).astype(bool).sum()),
                      "detected_pct_of_available": round(100 * have.detected.mean(), 1) if len(have) else np.nan,
                      "by_shape": int(ev.isin(["step", "gradual", "receding"]).sum()),
                      "by_presence": int(ev.isin(["appears", "goes_quiet"]).sum()),
                      "by_sparse_rate": int((ev == "sparse_rate_rise").sum())})

loss_rates = best.groupby("kind").apply(rate, include_groups=False)
loss_rates_ds = best.groupby(["kind", "dataset"]).apply(rate, include_groups=False)
display(loss_rates)
display(loss_rates_ds)

In [ ]:
# Per case: which kinds detect the shift (own latency / own error / best caller)
wide = best.assign(ev=np.where(best.detected, best.evidence, "-" + best.lean.astype(str))).pivot_table(
    index=["dataset", "case"], columns="kind", values="ev", aggfunc="first")
via = best[best.kind == "caller_latency"].set_index("case").via
wide["best_caller"] = wide.index.get_level_values("case").map(via)

det = best.pivot_table(index="case", columns="kind", values="detected", aggfunc="first").fillna(False).astype(bool)
print("own latency only:", (det.own_latency & ~det.caller_latency).sum(),
      "| caller only:", (~det.own_latency & det.caller_latency).sum(),
      "| both:", (det.own_latency & det.caller_latency).sum(),
      "| neither:", (~det.own_latency & ~det.caller_latency).sum(), f"(of {len(det)})")
neither = sorted(det.index[~det.own_latency & ~det.caller_latency])
weak = set(best[best.weak.fillna(False).astype(bool)].case)
print("neither (clear evidence):", [c + (" [weak]" if c in weak else "") for c in neither])
wide

### 4c. Where the filters hide subtle symptoms

Project principle: filtering and compression must not collapse subtle symptoms. `shape_signature_v2` has two filters worth checking directly:
- `dropna()`: a metric that **disappears** (service going quiet, no requests, so no latency/error samples) or **appears** after inject_time is invisible to it.
- **medians** (baseline level, late level): a sparse signal, such as errors in 20% of seconds, has median 0 and vanishes.

For every RE1/RE2 case, check the root-cause service's own columns for both.

In [ ]:
presence, errors = [], []
for _, r in idx[idx.suite.isin(["RE1", "RE2"])].iterrows():
    if r["case"] in EXCLUDE:
        continue
    m = load_metrics(r["case"])
    t = load_inject_time(r["case"])
    pre, post = m[m.time < t], m[m.time >= t]
    svc = r["root_cause_service"]
    for c in [c for c in m.columns if c.startswith(svc + "_")]:
        a, b = pre[c].notna().mean(), post[c].notna().mean()
        if abs(a - b) > 0.2:
            presence.append({"case": r["case"], "fault": r["fault"], "metric": c[len(svc) + 1:],
                             "pre_nonnull": round(a, 2), "post_nonnull": round(b, 2)})
    c = f"{svc}_error"
    if c in m.columns:
        pe, po = pre[c].fillna(0), post[c].fillna(0)
        late = po.values[-max(len(po) // 4, 1):]
        errors.append({"case": r["case"], "fault": r["fault"],
                       "pre_frac_nonzero": (pe > 0).mean(), "post_frac_nonzero": (po > 0).mean(),
                       "pre_mean": pe.mean(), "post_mean": po.mean(), "late_median": np.median(late)})
presence = pd.DataFrame(presence)
errors = pd.DataFrame(errors)

print(f"root-cause columns whose presence (non-null fraction) changes by >20 points across inject_time: {len(presence)}")
display(pd.crosstab(presence.fault, presence.metric))
presence["direction"] = np.where(presence.post_nonnull < presence.pre_nonnull, "goes quiet", "appears")
display(pd.crosstab([presence.fault, presence.metric], presence.direction))

In [ ]:
# Error rate rises (mean more than doubles, errors in >5% of fault seconds) that the median-based signature can't see
errors["rises"] = (errors.post_mean > 2 * errors.pre_mean + 1e-9) & (errors.post_frac_nonzero > 0.05)
errors["hidden_by_median"] = errors.rises & (errors.late_median == 0)
print(f"cases with an {{svc}}_error column: {len(errors)}")
display(errors.groupby("fault")[["rises", "hidden_by_median"]].sum())
errors[errors.hidden_by_median].round(3)

## 5. Log exploration scratchpad (carts case, for reference)

Reuse this section for the log-based (RE3) side of the investigation — narrow by container_name and time window interactively, without re-running a whole script each time.

In [ ]:
logs = pd.read_parquet(f"{DATA_DIR}/re3ss_carts_f1_1/logs.parquet")
inject = load_inject_time("re3ss_carts_f1_1")

window = logs[(logs.timestamp >= inject - 15) & (logs.timestamp <= inject + 60)]
window[window.container_name.isin(["carts", "carts-db"])]

## 6. RE3-SS `root_cause.txt` lines

8 RE3-SS cases ship a `root_cause.txt`: one CSV row `HH:MM, ns timestamp, container, message, pod, node` that marks a log line as the root-cause evidence. For each one:
1. Find it in `logs.parquet`. Collapse the message to a **template** (ids, numbers and timestamps replaced) so we can see whether it is a one-off or one instance of a repeated pattern, and when that pattern first appears.
2. Show ~10 lines of context from **all** services, around the marked line and around the pattern's first occurrence.
3. Compare what an `Exception|Error` keyword filter keeps vs drops.
4. See whether each fault label (f1-f4) has a consistent new log signature. Checked on all 30 RE3-SS cases, not just the 8.
5. Check each service's log volume for drops around inject_time (a service going quiet).

Caveat: `logs.parquet` timestamps are whole seconds (~60 lines/s here), so the order *within* a second is file order, not true order. "±10 lines" is therefore mostly the same second.

In [ ]:
import re
# read_root_cause, template, load_logs: rca_lib.py

rc_cases = idx[idx.has_root_cause_file][["case", "fault", "root_cause_service", "inject_time"]].reset_index(drop=True)
rc_logs = {c: load_logs(c) for c in rc_cases.case}
print("null messages per case:", {c: int(L.null_message.sum()) for c, L in rc_logs.items()})

rows = []
for _, r in rc_cases.iterrows():
    rc = read_root_cause(r.case)
    L = rc_logs[r.case]
    hit = L[(L.container_name == rc["container"]) & (L.message == rc["message"])]
    tm = template(rc["message"])
    same = L[(L.container_name == rc["container"]) & (L.tmpl == tm)]
    rows.append({
        "case": r.case, "fault": r.fault, "ground_truth": r.root_cause_service, "marked_container": rc["container"],
        "marked_message": rc["message"], "marked_rel_s": rc["ts"] - r.inject_time,
        "found_in_logs": len(hit), "parquet_ts_offset_s": (hit.timestamp.iloc[0] - rc["ts"]) if len(hit) else None,
        "template": tm, "template_count_pre": int((same.rel < 0).sum()), "template_count_post": int((same.rel >= 0).sum()),
        "template_first_rel_s": same.rel.min() if len(same) else None,
    })
rc_table = pd.DataFrame(rows)
dupes = rc_table[rc_table.duplicated("marked_message", keep=False)]
print("root_cause.txt files with identical content:", dupes.case.tolist())
rc_table

In [ ]:
def show_context(L, i, n=10):
    """n lines before/after row position i, all containers."""
    ctx = L.iloc[max(i - n, 0): i + n + 1]
    for j, row in ctx.iterrows():
        mark = ">>" if j == L.index[i] else "  "
        print(f"{mark} {row.rel:+5d}s  {row.container_name:13s} {row.message[:170]}")

for _, r in rc_table.iterrows():
    L = rc_logs[r.case]
    print(f"\n==================== {r.case}  (fault {r.fault}, ground truth: {r.ground_truth})")
    if not r.found_in_logs:
        print(f"marked line NOT FOUND in logs.parquet (marked at {r.marked_rel_s:+d}s vs inject_time)")
        continue
    pos = L.index.get_indexer(L[(L.container_name == r.marked_container) & (L.message == r.marked_message)].index)[0]
    print(f"--- around the MARKED line ({r.marked_rel_s:+d}s):")
    show_context(L, pos)
    first = L.index.get_indexer(L[(L.container_name == r.marked_container) & (L.tmpl == r.template)].index)[0]
    if first != pos:
        print(f"--- around the FIRST occurrence of the same template ({L.iloc[first].rel:+d}s):")
        show_context(L, first)

### 6b. What an `Exception|Error` keyword filter keeps vs drops

The filter is applied case-sensitively as written (`Exception|Error`), and also case-insensitively (`(?i)exception|error`), which additionally matches `error` inside words and paths. For each case: is the marked line kept? Are the other instances of its template kept? What does the filter keep instead? The kept lines are what a model would see under this filter.

In [ ]:
FILTERS = {"Exception|Error": re.compile(r"Exception|Error"), "(?i)exception|error": re.compile(r"(?i)exception|error")}

kd = []
for _, r in rc_table.iterrows():
    L = rc_logs[r.case]
    post = L[L.rel >= 0]
    tmpl_rows = post[(post.container_name == r.marked_container) & (post.tmpl == r.template)]
    for name, rx in FILTERS.items():
        keep_post = post.message.str.contains(rx, na=False)
        keep_pre = L[L.rel < 0].message.str.contains(rx, na=False)
        kd.append({"case": r.case, "filter": name,
                   "marked_line_kept": bool(rx.search(r.marked_message)),
                   "template_instances_kept": f"{int(tmpl_rows.message.str.contains(rx, na=False).sum())}/{len(tmpl_rows)}",
                   "fault_lines_kept": f"{int(keep_post.sum())}/{len(post)}",
                   "baseline_lines_kept": f"{int(keep_pre.sum())}/{int((L.rel < 0).sum())}",
                   "kept_by_container_fault": post[keep_post].container_name.value_counts().to_dict()})
pd.DataFrame(kd)

In [ ]:
# What the case-sensitive filter feeds a model during the fault period: top kept templates per case,
# with how often the same template also appears in the normal period (baseline noise vs new).
rx = FILTERS["Exception|Error"]
for _, r in rc_table.iterrows():
    L = rc_logs[r.case]
    kept = L[L.message.str.contains(rx, na=False)]
    pre_counts = kept[kept.rel < 0].groupby(["container_name", "tmpl"]).size()
    top = kept[kept.rel >= 0].groupby(["container_name", "tmpl"]).size().sort_values(ascending=False).head(6)
    print(f"\n=== {r.case} (ground truth {r.ground_truth}) - top kept templates in the fault period [post / pre]")
    for (c, t), n in top.items():
        print(f"  {n:4d} / {pre_counts.get((c, t), 0):4d}  {c:12s} {t[:150]}")

### 6c. Do f1-f4 have consistent log signatures?

For every RE3-SS case (all 30, whether or not it has `root_cause.txt`), list the **new** templates: those appearing ≥3 times after inject_time and never before. Group by fault label and service to see whether a label maps to one kind of signature.

In [ ]:
re3ss = idx[idx.dataset == "RE3-SS"].sort_values(["fault", "root_cause_service", "case"])
new_sig = []
for _, r in re3ss.iterrows():
    L = rc_logs.get(r.case)
    if L is None:
        L = load_logs(r.case)
    pre_t = set(zip(L[L.rel < 0].container_name, L[L.rel < 0].tmpl))
    post = L[L.rel >= 0]
    counts = post.groupby(["container_name", "tmpl"]).agg(n=("rel", "size"), first_rel=("rel", "min"))
    counts = counts[np.array([k not in pre_t for k in counts.index], dtype=bool) & (counts.n >= 3).values].sort_values("n", ascending=False)
    for (c, t), row in counts.head(4).iterrows():
        new_sig.append({"fault": r.fault, "service": r.root_cause_service, "case": r.case,
                        "has_rc_file": r.has_root_cause_file, "container": c, "n": row.n,
                        "first_rel_s": row.first_rel, "new_template": t[:140]})
    if counts.empty:
        new_sig.append({"fault": r.fault, "service": r.root_cause_service, "case": r.case,
                        "has_rc_file": r.has_root_cause_file, "container": "-", "n": 0, "first_rel_s": None,
                        "new_template": "(no new template with >=3 occurrences)"})
new_sig = pd.DataFrame(new_sig)
new_sig

### 6d. Log volume around inject_time: does any service go quiet?

Lines per 10 s per container. A container is flagged if some 30 s stretch after inject_time runs at < 25% of its median pre-inject rate (containers with < 0.2 lines/s at baseline are too sparse to judge, so they're listed separately rather than dropped).

In [ ]:
# quiet_periods: rca_lib.py
quiet = []
vols = {}
for _, r in rc_cases.iterrows():
    q, vols[r.case] = quiet_periods(rc_logs[r.case])
    q.insert(0, "case", r.case)
    quiet.append(q)
quiet = pd.concat(quiet, ignore_index=True)
print("containers that go quiet after inject_time:")
display(quiet[quiet.quiet_from_s.notna()])
print("containers too sparse to judge (baseline < 0.2 lines/s):")
quiet[quiet.sparse_baseline].groupby("case").container.apply(list)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 8))
for ax, (_, r) in zip(axes.flat, rc_cases.iterrows()):
    v = vols[r.case]
    for c in v.index:
        lw = 2 if c == r.root_cause_service else 0.8
        ax.plot(v.columns, v.loc[c] + 0.5, lw=lw, label=c)
    ax.set_yscale("log")
    ax.axvline(0, color="red", lw=1)
    ax.set_title(f"{r.case} (gt {r.root_cause_service}) - lines per 10 s", fontsize=9)
axes.flat[0].legend(fontsize=6, ncol=2)
plt.tight_layout()
plt.show()

## 7. Diagnosability and scoring exclusions

A case is **not diagnosable from available data** when it has no logs and no traces, and no metric of the root-cause service or of its known callers shows any shift. "Shift" means any of: a clear shape change, a presence change (metric appears or goes quiet), or a sparse-rate rise (errors). Cases whose only evidence is **borderline** are listed separately as **weak evidence only**. They are kept for scoring but should be read with that in mind.

Scope of "any metric": the root-cause service's own columns plus its callers' latency. Other services' metrics are *not* used to rescue a case: in a system with 50-370 metrics, some unrelated metric almost always moves. For transparency, the table still counts how many other-service metrics shift.

RE3 carts f4 cases have logs but no new log pattern (6c). They get the same metric check, plus a log check for **rate** changes in existing patterns and per-container volume (novelty alone would miss a rate-only symptom, as front-end f1 showed).

In [ ]:
# service_evidence, diagnosability_status: rca_lib.py
diag_rows, diag_detail = [], {}
check = idx[(~idx.has_logs & ~idx.has_traces) | ((idx.dataset == "RE3-SS") & (idx.root_cause_service == "carts") & (idx.fault == "f4"))]
for _, r in check.iterrows():
    if r["case"] in EXCLUDE:
        continue
    m = load_metrics(r["case"])
    t = load_inject_time(r["case"])
    ev = service_evidence(r["case"], r["system"], r["root_cause_service"], t, m)
    diag_detail[r["case"]] = ev
    status = diagnosability_status(ev)
    diag_rows.append({"case": r["case"], "dataset": r["dataset"], "fault": r["fault"], "service": r["root_cause_service"],
                      "status": status, "metrics_checked": len(ev),
                      "evidence": "; ".join(f"{x.metric}:{x.evidence}" for x in ev.itertuples() if x.evidence)})
diag = pd.DataFrame(diag_rows)
pd.crosstab(diag.dataset, diag.status)

In [ ]:
# Transparency: for cases without clear own/caller evidence, how many OTHER-service metrics shift anyway?
unclear = diag[diag.status != "diagnosable"].copy()
def other_shifts(case, svc, system):
    m = load_metrics(case)
    t = load_inject_time(case)
    skip = {svc} | {c for c, _ in callers_of(system, svc)}
    cols = [c for c in m.columns if c != "time" and c.rsplit("_", 1)[0] not in skip]
    hits = [c for c in cols if metric_evidence(m, c, t)["shifted"]]
    return f"{len(hits)}/{len(cols)}", ", ".join(hits[:6]) + (" ..." if len(hits) > 6 else "")
unclear[["other_metrics_shifted", "examples"]] = [other_shifts(c, s, idx.set_index("case").loc[c, "system"])
                                                   for c, s in zip(unclear.case, unclear.service)]
unclear[["case", "status", "metrics_checked", "evidence", "other_metrics_shifted", "examples"]]

In [ ]:
# RE3-SS carts f4: logs exist but no new pattern. Check rate changes of existing patterns and container volume
# (log_rate_changes: rca_lib.py).
for case in diag[diag.case.str.startswith("re3ss_carts_f4")].case:
    changes, vol_ratio = log_rate_changes(case)
    print(f"\n=== {case}: metric status = {diag.set_index('case').loc[case, 'status']}")
    print("   log pattern rate changes (>=3x, >=20 lines):", "none" if changes.empty else "")
    for (c, tm), row in changes.iterrows():
        print(f"     {c:12s} pre={int(row.pre):4d} post={int(row.post):4d} x{row.ratio_post_over_pre}  {tm[:110]}")
    print("   container volume post/pre:", vol_ratio.to_dict())

In [ ]:
# Scoring exclusions: everything that must stay out of model scoring, with the reason. Not written to disk yet.
SCORING_EXCLUSIONS = {**EXCLUDE, **BROKEN_LABELS}  # static part: rca_lib.py
for c in diag[diag.status == "not diagnosable from available data"].case:
    if c.startswith("re3ss_carts_f4"):
        changes, _ = log_rate_changes(c)
        if not changes.empty:
            continue  # logs still show a rate change - keep it
    SCORING_EXCLUSIONS[c] = "not diagnosable from available data: no shift in own/caller metrics" + (
        ", no new log pattern and no log rate change" if c.startswith("re3") else ", no logs or traces")

print(f"{len(SCORING_EXCLUSIONS)} cases excluded from scoring:")
pd.Series(SCORING_EXCLUSIONS, name="reason").rename_axis("case").to_frame()

### Known limitations (record, not fixed)

- **Caller map gaps.** Sock Shop has no traces anywhere, so its caller edges come from the published architecture, not from this data. OB adservice / cartservice / shippingservice never appear in OB traces, so their callers are from the architecture too. **ts-auth-service has no callers** in TT traces and none were added: for TT auth cases there is no caller evidence at all. Trace edges are sampled from 10 cases per dataset.
- **`re3ss_front-end_f2_2`'s `root_cause.txt` is broken.** It duplicates f2_1's file (its line isn't in this case's logs). Excluded from scoring.
- **`{svc}_diskio` appearing after injection** happens almost only on the root-cause service (17/18 cases), including RE3 code-fault cases. That suggests it is at least partly an artifact of redeploying/restarting the faulty service, not only a disk-fault symptom. It is counted as evidence (it is real data), but a method that relies on it may be exploiting the injection mechanism.
- **Log timestamps are whole seconds** (~60 lines/s). Order within a second is file order, not causal order.
- **Templates** keep HTTP status codes but still merge other numbers (e.g. response sizes 64 vs 70 bytes).
- **The RE3 fault labels are defined in the paper, not in the data**: F1 incorrect parameter values, F2 missing parameters, F3 missing function call, F4 incorrect return values, F5 missing exception handlers (RCAEval, arXiv 2412.17015). Which code was changed in which service isn't published.
- **"Not diagnosable" is relative to this check**: own + caller metrics with the current provisional thresholds. A better detector could move cases out of it.

### Scoring rules (decided 2026-09-22)

- **Weak-evidence-only cases stay in scoring**, but every score is reported separately for **clear** and **weak** cases (never only a combined number).
- **`diskio_appears` is a possible injection artifact.** It stays in the evidence with `artifact_flag`, and every scoring run is done **with and without** artifact-flagged evidence.
- `SCORING_EXCLUSIONS` (above) are never scored.

### Step 0 configurations vs hardware (decided 2026-09-22)

- **Unranked** (nothing cut, `num_ctx` per case): OB and SS only.
- **TrainTicket unranked: not runnable on 6 GB VRAM.** RE3-TT reaches ~11.8k tokens (`num_ctx` ~16k), and RE2-TT ~7k; that context does not fit alongside the model on this GPU, and gemma4:26b already spills to CPU.
- **Capped** (~3k tokens with a swappable ranking step): all datasets, including TT.
- **Measured on this machine (2026-09-22), which contradicts the 6 GB premise:** the GPU is an RTX A4500 with **20 GB**, and qwen2.5-coder:7b stays **100% on GPU at every context size tested** (4k: 4.53 GB, 8k: 4.90 GB, 11k: 5.07 GB, 16k: 5.36 GB), ~92-94 tok/s throughout. On this GPU, TT unranked (~16k) would fit. The "6 GB" limit above came from the stated hardware and is pending confirmation; gemma4:26b (18 GB) was not measured and will be much tighter.

### Parked for later (not done yet)

- **Position-bias test**: the step-0 text lists services alphabetically, so the root cause is sometimes first by accident (carts in Sock Shop). Re-run scoring with the service order shuffled and compare.
- **No-`t=0` test**: step 0 tells the model when the fault started (the standard RCAEval setting, but a strong hint). Re-run without it.

## 8. Step 0: compressed evidence per case (`rca_lib.render_step0`, nothing written to `derived/`)

Per case, a **ground-truth-blind** text block for a model. It covers every service in alphabetical order, with no case name, fault label or root-cause marker. `t=0` is inject_time.

- **Metrics**: clear changes get one row each (shape / presence / error-seconds, size, when it settled). Weak changes are listed compactly per service. `diskio_appears` rows are marked as a possible injection artifact and can be excluded (`include_artifacts=False`).
- **Logs**: per-service volume and quiet periods. Patterns that are new, vanished, or changed rate ≥3× (clear) or 2-3× (weak), with no keyword filter and HTTP status codes kept. One-off new lines and rare events (≤3 lines in total, seen before *and* after t=0, e.g. a restart banner) are summarised with count, time span and examples **sampled across the span**. Exception/error class names are always listed and never truncated.
- **Two configurations (decided)**: *unranked*: nothing cut, and Ollama `num_ctx` is set per case with `num_ctx_for(prompt tokens)` (6 GB VRAM, so no flat 16k). *Capped*: a ranking/narrowing step to fit ~3k tokens. That one is a swappable step 1 and is not built yet. Every model call checks `prompt_eval_count` with `check_prompt_eval`.

In [ ]:
step0_example = render_step0("re3ss_carts_f1_1")
print(step0_example)

In [ ]:
# Token counts: qwen2.5-coder exact (its tokenizer), gemma4 approximate (Gemma 3 tokenizer); num_ctx per case.
sizes = []
for case, kw in [("re3ss_carts_f1_1", {}), ("re3ss_carts_f1_1", {"max_pat_rows": 5}), ("re3ss_carts_f1_1", {"include_artifacts": False}),
                 ("re1ob_adservice_mem_1", {}), ("re1tt_ts-order-service_cpu_1", {}), ("re2ss_carts_loss_1", {}),
                 ("re2ob_checkoutservice_cpu_1", {}), ("re2tt_ts-order-service_delay_1", {}), ("re3tt_ts-auth-service_f1_1", {})]:
    txt = render_step0(case, **kw)
    q, g = count_tokens(txt, "qwen"), count_tokens(txt, "gemma")
    sizes.append({"case": case, "options": kw, "qwen_tokens": q, "gemma_tokens_approx": g, "num_ctx": num_ctx_for(max(q, g)),
                  "WARN kept": ("Request method 'POST' not supported" in txt) if case == "re3ss_carts_f1_1" else None,
                  "omitted": "; ".join(l for l in txt.splitlines() if l.startswith("OMITTED"))})
pd.DataFrame(sizes)